# Horovod Distributed Training with SageMaker TensorFlow script mode (SageMaker Python SDK V3)

---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-script_mode_distributed_training_horovod_tensorflow|sm-script_mode_distributed_training_horovod_tensorflow.ipynb)

---

Horovod is a distributed training framework based on Message Passing Interface (MPI). For information about Horovod, see [Horovod README](https://github.com/uber/horovod).

You can perform distributed training with Horovod on SageMaker by using the SageMaker TensorFlow container. If MPI is enabled when you create the training job, SageMaker creates the MPI environment and executes the `mpirun` command to execute the training script.

This notebook has been migrated to the **SageMaker Python SDK V3**. In V3, the framework-specific `TensorFlow` estimator is replaced by the generic `ModelTrainer`, and the MPI/Horovod launch is configured with the `sagemaker.train.distributed.MPI` config object (which drives `mpirun` in the backend, exactly like the V2 `distribution={"mpi": {...}}` dictionary).

In this example notebook, we create a Horovod training job that uses the MNIST data set.

## Set up the environment

We get the `IAM` role that this notebook is running as and pass that role to the `ModelTrainer` that SageMaker uses to get data and perform training.

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer, Mode
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.train.distributed import MPI
from sagemaker.core.training.configs import InputData, Networking
from sagemaker.core.shapes import StoppingCondition
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sagemaker_session = Session()
region = sagemaker_session.boto_region_name

default_s3_bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
default_bucket_prefix_path = ""

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    default_bucket_prefix_path = f"/{default_bucket_prefix}"

sagemaker_iam_role = get_execution_role()

train_script = "mnist_hvd.py"
instance_count = 2

## Prepare Data for training

Now we download the MNIST dataset to the local `/tmp/data/` directory and then upload it to an S3 bucket. After uploading the dataset to S3, we delete the data from `/tmp/data/`. 

In [ ]:
import os
import shutil

import numpy as np

import keras
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

s3_train_path = "s3://{}{}/mnist/train.npz".format(default_s3_bucket, default_bucket_prefix_path)
s3_test_path = "s3://{}{}/mnist/test.npz".format(default_s3_bucket, default_bucket_prefix_path)

# Create local directory
! mkdir -p /tmp/data/mnist_train
! mkdir -p /tmp/data/mnist_test

# Save data locally
np.savez("/tmp/data/mnist_train/train.npz", data=x_train, labels=y_train)
np.savez("/tmp/data/mnist_test/test.npz", data=x_test, labels=y_test)

# Upload the dataset to s3
! aws s3 cp /tmp/data/mnist_train/train.npz $s3_train_path
! aws s3 cp /tmp/data/mnist_test/test.npz $s3_test_path

print("training data at ", s3_train_path)
print("test data at ", s3_test_path)
! rm -rf /tmp/data

## Write a script for horovod distributed training

This example is based on the [Keras MNIST horovod example](https://github.com/uber/horovod/blob/master/examples/keras_mnist.py) example in the horovod github repository.

To run this script we have to make following modifications:

### 1. Accept `--model_dir` as a command-line argument
Modify the script to accept `model_dir` as a command-line argument that defines the directory path (i.e. `/opt/ml/model/`) where the output model is saved. Because Sagemaker deletes the training cluster when training completes, saving the model to `/opt/ml/model/` directory prevents the trained model from getting lost, because when the training job completes, SageMaker writes the data stored in `/opt/ml/model/` to an S3 bucket. 

This also allows the SageMaker training job to integrate with other SageMaker services, such as hosted inference endpoints or batch transform jobs. It also allows you to host the trained model outside of SageMaker.

The following code adds `model_dir` as a command-line argument to the script:

```
parser = argparse.ArgumentParser()
parser.add_argument('--model_dir', type=str)
```

### 2. Load train and test data

You can get local directory path where the `train` and `test` data is downloaded by reading the environment variable `SM_CHANNEL_TRAIN` and `SM_CHANNEL_TEST` respectively.
After you get the directory path, load the data into memory.

Here is the code:

```
x_train = np.load(os.path.join(os.environ['SM_CHANNEL_TRAIN'], 'train.npz'))['data']
y_train = np.load(os.path.join(os.environ['SM_CHANNEL_TRAIN'], 'train.npz'))['labels']

x_test = np.load(os.path.join(os.environ['SM_CHANNEL_TEST'], 'test.npz'))['data']
y_test = np.load(os.path.join(os.environ['SM_CHANNEL_TEST'], 'test.npz'))['labels']
```

The `channel_name` you pass to `InputData` (`train` / `test`) becomes the `SM_CHANNEL_<NAME>` environment variable inside the training container.

### 3. Save the model only at the master node

Because in Horovod the training is distributed to multiple nodes, the model should only be saved by the master node. The following code in the script does this:

```
# Horovod: Save model only on worker 0 (i.e. master)
if hvd.rank() == 0:
    model.save(os.path.join(args.model_dir, "model.h5"))
```

### Training script

Here is the final training script. The training script itself is framework code (Horovod + TensorFlow) and is unchanged by the V2 to V3 SDK migration.

In [ ]:
!cat 'mnist_hvd.py'

## Retrieve the TensorFlow training image

In V3 the generic `ModelTrainer` needs an explicit training image. We use `image_uris.retrieve` to get the same TensorFlow container that the V2 `TensorFlow` estimator selected automatically via `framework_version="1.15.2"` / `py_version="py3"`.

In [ ]:
train_instance_type = "ml.c4.xlarge"

# Note: py_version is "py37" (not "py3"). Both are TensorFlow 1.15.2, but the V3
# ModelTrainer MPI/Horovod launcher runs inside the container and uses
# subprocess.run(capture_output=...), which requires Python >= 3.7. The "py3" build
# of this image ships Python 3.6, so "py37" is required for distributed MPI training.
training_image = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="1.15.2",
    py_version="py37",
    instance_type=train_instance_type,
    image_scope="training",
)
print(training_image)

source_code = SourceCode(
    source_dir=".",
    entry_script=train_script,
)

## Test locally using ModelTrainer local container mode (reference only)

This notebook shows how to run your code in a local container before deploying to SageMaker's managed training environment. In V2 this was done by setting `train_instance_type="local"`. In V3 you set `training_mode=Mode.LOCAL_CONTAINER` on the `ModelTrainer`.

> **Note:** Local mode requires a local Docker (and, for multi-node MPI/Horovod, additional container networking) that is not available in every environment. The cells below show the V3 local-mode configuration for reference and are not executed as part of the end-to-end validation of this notebook. To run locally, uncomment and run them on a machine with Docker installed. You can only run a single local notebook at a time.

The MPI environment for Horovod is configured by passing a `sagemaker.train.distributed.MPI` object to the `distributed` argument of `ModelTrainer`:

* Creating an `MPI(...)` object is equivalent to `{"mpi": {"enabled": True}}` in V2 (the MPI setup is performed and the `mpirun` command is executed).
* `process_count_per_node (int) [Optional]`: Number of processes MPI should launch on each host. Equivalent to `processes_per_host` in V2. Should not be greater than the available slots on the selected instance type.
* `mpi_additional_options (List[str]) [Optional]`: Any `mpirun` flag(s) that will be added to the `mpirun` command executed by SageMaker. Equivalent to `custom_mpi_options` in V2.

In [ ]:
# Reference only - requires local Docker; not run during e2e validation.
#
# estimator_local = ModelTrainer(
#     training_mode=Mode.LOCAL_CONTAINER,
#     training_image=training_image,
#     role=sagemaker_iam_role,
#     source_code=source_code,
#     compute=Compute(instance_count=instance_count, instance_type="local"),
#     distributed=MPI(),
#     hyperparameters={"model_dir": "/opt/ml/model"},
#     base_job_name="hvd-mnist-local",
# )
#
# estimator_local.train(
#     input_data_config=[
#         InputData(channel_name="train", data_source=s3_train_path),
#         InputData(channel_name="test", data_source=s3_test_path),
#     ]
# )

## Train in SageMaker

After you test the training job locally, run it on SageMaker using a valid EC2 instance type such as `ml.c4.xlarge`.

You can also provide your custom MPI options by passing the `mpi_additional_options` list to the `MPI` config; these are added to the `mpirun` command executed by SageMaker (equivalent to `custom_mpi_options` in V2).

In [ ]:
# custom mpirun options are passed as a list of strings via mpi_additional_options
# (equivalent to the V2 custom_mpi_options string). Note: environment variables must be
# passed to mpirun with the `-x` flag (e.g. `-x NCCL_DEBUG=INFO`); `--NCCL_DEBUG=INFO`
# is not a valid mpirun option.
distribution = MPI(mpi_additional_options=["-verbose", "-x", "NCCL_DEBUG=INFO"])

# The training script reads --model_dir to know where to save the model. In V2 the
# TensorFlow estimator injected this argument automatically; with the generic V3
# ModelTrainer we pass it explicitly as a hyperparameter (the launcher turns
# {"model_dir": ...} into the CLI argument --model_dir). /opt/ml/model is the standard
# container path that SageMaker uploads to S3 as the model artifact.
estimator = ModelTrainer(
    training_image=training_image,
    role=sagemaker_iam_role,
    source_code=source_code,
    compute=Compute(instance_count=instance_count, instance_type=train_instance_type),
    distributed=distribution,
    hyperparameters={"model_dir": "/opt/ml/model"},
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    base_job_name="hvd-mnist",
)

Call `train()` to start the training. The `channel_name` of each `InputData` (`train` / `test`) maps to the `SM_CHANNEL_TRAIN` / `SM_CHANNEL_TEST` environment variables read by the training script.

In [ ]:
estimator.train(
    input_data_config=[
        InputData(channel_name="train", data_source=s3_train_path),
        InputData(channel_name="test", data_source=s3_test_path),
    ]
)

##  Horovod training in SageMaker using multiple CPU/GPU

To enable multiple CPUs or GPUs for horovod training, set the `process_count_per_node` field of the `MPI` config to the desired number of processes to run per instance (equivalent to `processes_per_host` in V2).

In [ ]:
distribution = MPI(process_count_per_node=2)

estimator = ModelTrainer(
    training_image=training_image,
    role=sagemaker_iam_role,
    source_code=source_code,
    compute=Compute(instance_count=instance_count, instance_type=train_instance_type),
    distributed=distribution,
    hyperparameters={"model_dir": "/opt/ml/model"},
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    base_job_name="hvd-mnist-multi-cpu",
)

Call `train()` to start the training

In [ ]:
estimator.train(
    input_data_config=[
        InputData(channel_name="train", data_source=s3_train_path),
        InputData(channel_name="test", data_source=s3_test_path),
    ]
)

## Improving horovod training performance on SageMaker (reference only)

Performing Horovod training inside a VPC improves the network latency between nodes, leading to higher performance and stability of Horovod training jobs.

> **Note:** The VPC section below creates real AWS networking resources (a CloudFormation stack with a VPC, subnets, security group, S3 VPC endpoint, and route table) and is a performance optimization on top of the standard training shown above. It is provided here for reference and is not executed as part of the end-to-end validation of this notebook. To use it, uncomment the cells and run them in an account where you have permission to create the networking resources.

### Setup VPC infrastructure
We will setup following resources as part of VPC stack:
* `VPC`: AWS Virtual private cloud with CIDR block.
* `Subnets`: Two subnets with the CIDR blocks `10.0.0.0/24` and `10.0.1.0/24`
* `Security Group`: Defining the open ingress and egress ports, such as TCP.
* `VpcEndpoint`: S3 Vpc endpoint allowing sagemaker's vpc cluster to download data from S3.
* `Route Table`: Defining routes and is tied to subnets and VPC.

Complete cloud formation template for setting up the VPC stack can be seen [here](./vpc_infra_cfn.json).

In [ ]:
# Reference only - creates real VPC/networking resources; not run during e2e validation.
#
# import boto3
# from botocore.exceptions import ClientError
# from time import sleep
#
#
# def create_vpn_infra(stack_name="hvdvpcstack"):
#     cfn = boto3.client("cloudformation")
#
#     cfn_template = open("vpc_infra_cfn.json", "r").read()
#
#     try:
#         vpn_stack = cfn.create_stack(StackName=(stack_name), TemplateBody=cfn_template)
#     except ClientError as e:
#         if e.response["Error"]["Code"] == "AlreadyExistsException":
#             print("Stack: {} already exists, so skipping stack creation.".format(stack_name))
#         else:
#             print("Unexpected error: %s" % e)
#             raise e
#
#     describe_stack = cfn.describe_stacks(StackName=stack_name)["Stacks"][0]
#
#     while describe_stack["StackStatus"] == "CREATE_IN_PROGRESS":
#         describe_stack = cfn.describe_stacks(StackName=stack_name)["Stacks"][0]
#         sleep(0.5)
#
#     if describe_stack["StackStatus"] != "CREATE_COMPLETE":
#         raise ValueError("Stack creation failed in state: {}".format(describe_stack["StackStatus"]))
#
#     print(
#         "Stack: {} created successfully with status: {}".format(
#             stack_name, describe_stack["StackStatus"]
#         )
#     )
#
#     subnets = []
#     security_groups = []
#
#     for output_field in describe_stack["Outputs"]:
#
#         if output_field["OutputKey"] == "SecurityGroupId":
#             security_groups.append(output_field["OutputValue"])
#         if output_field["OutputKey"] == "Subnet1Id" or output_field["OutputKey"] == "Subnet2Id":
#             subnets.append(output_field["OutputValue"])
#
#     return subnets, security_groups
#
#
# subnets, security_groups = create_vpn_infra()
# print("Subnets: {}".format(subnets))
# print("Security Groups: {}".format(security_groups))

### VPC training in SageMaker
Now, we create the `ModelTrainer`, passing the `compute`, `distributed`, and `networking` configuration. In V3 the VPC `subnets` and `security_group_ids` are provided via the `Networking` config object (they were passed directly to the V2 `TensorFlow` estimator).

In [ ]:
# Reference only - depends on the VPC resources created above; not run during e2e validation.
#
# estimator = ModelTrainer(
#     training_image=training_image,
#     role=sagemaker_iam_role,
#     source_code=source_code,
#     compute=Compute(instance_count=instance_count, instance_type=train_instance_type),
#     distributed=distribution,
#     networking=Networking(security_group_ids=security_groups, subnets=subnets),
#     hyperparameters={"model_dir": "/opt/ml/model"},
#     stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
#     base_job_name="hvd-mnist-vpc",
# )
#
# estimator.train(
#     input_data_config=[
#         InputData(channel_name="train", data_source=s3_train_path),
#         InputData(channel_name="test", data_source=s3_test_path),
#     ]
# )

After training is completed, you can host the saved model by using TensorFlow Serving on SageMaker.

## Reference Links:
* [Horovod Official Documentation](https://github.com/uber/horovod)
* [SageMaker Python SDK V3 ModelTrainer distributed training](https://github.com/aws/sagemaker-python-sdk)

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-script_mode_distributed_training_horovod_tensorflow|sm-script_mode_distributed_training_horovod_tensorflow.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-script_mode_distributed_training_horovod_tensorflow|sm-script_mode_distributed_training_horovod_tensorflow.ipynb)
